# 02. Model Training and Evaluation

This notebook demonstrates how to train an `E3GNN` model and evaluate its performance.

In [1]:
import torch
import pytorch_lightning as pl
from pathlib import Path
from omegaconf import OmegaConf
from dataclasses import asdict

from data.factory import DatasetFactory
from net.common import HyperParams
from net.e3gnn import E3GNN

/home/bartek/casus/mandala/mandala-venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Creating a Dataset

We use a `DatasetFactory` to create the training and validation datasets.

In [3]:
# Create a DatasetFactory
factory = DatasetFactory(
    cutoff_gnn=4.0,
    cutoff_matrix=6.0,
    l_max_sh=2,
    n_radial=64,
)

# Add snapshots to the factory
factory.add_snapshot(
    "../../data/big/silicon/900K/Si_DM",
    "../../data/big/silicon/900K/info.txt",
    purpose="train",
)
factory.add_snapshot(
    "../../data/big/silicon/2700K/Si_DM",
    "../../data/big/silicon/2700K/info.txt",
    purpose="val",
)

# Create the datasets and the mapper
train_ds, val_ds, mapper = factory.create()

print("Training dataset size:", len(train_ds))
print("Validation dataset size:", len(val_ds))

/home/bartek/casus/mandala/mandala-venv/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Loading snapshots: 100%|██████████| 1/1 [00:08<00:00,  8.33s/it]

Training dataset size: 1
Validation dataset size: 1


## Model Initialization

We instantiate an `E3GNN` model with a minimal configuration for this demo.

In [4]:
# Model hyperparameters
hp = HyperParams(
    hidden_base_dim=32,
    l_max=2,
    num_layers_gnn=2,
    num_layers_matrix=1,
)

# Mock OmegaConf config
mock_cfg = OmegaConf.create(
    {"model": asdict(hp), "training": {"lr": 1e-3}, "logging": {"pedantic": False}}
)

# Create the model
model = E3GNN(mapper, train_ds.edge_types, mock_cfg)

print(model)

E3GNN(
  (node_enc): NodeEncoder(
    (elem_emb): Embedding(1, 32)
    (lin): Linear(32x0e -> 32x0e+32x0o+16x1e+16x1o+8x2e+8x2o | 1024 weights)
    (nl): Sequential(
      (0): Identity()
      (1): NormActivation(
        (norm): Norm(32x0e+32x0o+16x1e+16x1o+8x2e+8x2o)
        (scalar_nonlinearity): SiLU()
        (scalar_multiplier): ElementwiseTensorProduct(112x0e x 32x0e+32x0o+16x1e+16x1o+8x2e+8x2o -> 32x0e+32x0o+16x1e+16x1o+8x2e+8x2o | 112 paths | 0 weights)
      )
    )
    (dropout): Identity()
  )
  (edge_enc): EdgeEncoder(
    (edge_emb): Embedding(1, 32)
    (radial_net): RadialMLP(
      (net): Sequential(
        (0): Linear(in_features=64, out_features=128, bias=True)
        (1): SiLU()
        (2): Linear(in_features=128, out_features=32, bias=True)
      )
    )
    (lin_scalar): Linear(64x0e -> 32x0e+32x0o+16x1e+16x1o+8x2e+8x2o | 2048 weights)
    (sh_proj): Linear(1x0e+1x1o+1x2e -> 32x0e+32x0o+16x1e+16x1o+8x2e+8x2o | 56 weights)
    (nl): Sequential(
      (0): Ident

## Training the Model

We use the PyTorch Lightning `Trainer` to run the training loop.

In [7]:
# Create DataLoaders
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=1, collate_fn=lambda b: b[0])
val_dl = torch.utils.data.DataLoader(val_ds, batch_size=1, collate_fn=lambda b: b[0])

# Create a Trainer
trainer = pl.Trainer(max_epochs=30, accelerator="cpu", devices=1)

# Run training
trainer.fit(model, train_dl, val_dl)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
/home/bartek/casus/mandala/mandala-venv/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name     | Type        | Params | Mode 
-------------------------------------------------
0 | node_enc | NodeEncoder | 1.1 K  | train
1 | edge_enc | EdgeEncoder | 14.6 K | train
2 | mp_small | ModuleList  | 21.5 K | train
3 | mp_large | ModuleList  | 10.8 K | train
4 | heads    | ModuleDict  | 10.1 K | train
-------------------------------------------------
58.0 K    Trainable params
0         Non-trainable params
58.0 K    Total params
0.232     Total estimated model params size (MB)
240       Modules in train mode
0         Modules in eval mode


Sanity Checking DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

/home/bartek/casus/mandala/mandala-venv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


/home/bartek/casus/mandala/mandala-venv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/home/bartek/casus/mandala/mandala-venv/lib/python3.10/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 29: 100%|██████████| 1/1 [00:05<00:00,  0.18it/s, v_num=1, train_loss_step=3.8e+11, train_loss_blocks_step=6.12e+3, train_loss_E_step=4.37e+15, train_loss_N_step=3.36e+16, train_abs_error_E_step=6.61e+7, train_abs_error_N_step=1.83e+8, val_loss_step=3.41e+11, val_loss_blocks_step=4.51e+3, val_loss_E_step=4.71e+15, val_loss_N_step=2.94e+16, val_abs_error_E_step=6.86e+7, val_abs_error_N_step=1.71e+8, val_loss_epoch=3.41e+11, val_loss_blocks_epoch=4.51e+3, val_loss_E_epoch=4.71e+15, val_loss_N_epoch=2.94e+16, val_abs_error_E_epoch=6.86e+7, val_abs_error_N_epoch=1.71e+8, train_loss_epoch=3.8e+11, train_loss_blocks_epoch=6.12e+3, train_loss_E_epoch=4.37e+15, train_loss_N_epoch=3.36e+16, train_abs_error_E_epoch=6.61e+7, train_abs_error_N_epoch=1.83e+8]  

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|██████████| 1/1 [00:05<00:00,  0.18it/s, v_num=1, train_loss_step=3.8e+11, train_loss_blocks_step=6.12e+3, train_loss_E_step=4.37e+15, train_loss_N_step=3.36e+16, train_abs_error_E_step=6.61e+7, train_abs_error_N_step=1.83e+8, val_loss_step=3.41e+11, val_loss_blocks_step=4.51e+3, val_loss_E_step=4.71e+15, val_loss_N_step=2.94e+16, val_abs_error_E_step=6.86e+7, val_abs_error_N_step=1.71e+8, val_loss_epoch=3.41e+11, val_loss_blocks_epoch=4.51e+3, val_loss_E_epoch=4.71e+15, val_loss_N_epoch=2.94e+16, val_abs_error_E_epoch=6.86e+7, val_abs_error_N_epoch=1.71e+8, train_loss_epoch=3.8e+11, train_loss_blocks_epoch=6.12e+3, train_loss_E_epoch=4.37e+15, train_loss_N_epoch=3.36e+16, train_abs_error_E_epoch=6.61e+7, train_abs_error_N_epoch=1.83e+8]


## Making Predictions

We can use the trained model to make predictions on a new snapshot.

In [8]:
# Get a sample from the validation set
x_gnn, x_mat, y_true = val_ds[0]

# Make a prediction
y_pred = model(x_gnn, x_mat)

# Compare the predicted energy to the true energy
true_energy = y_true["energy"].item()
pred_snap = model.predictions_to_snapshot(y_pred, x_gnn["positions"], x_gnn["box"])
pred_energy = pred_snap.get_energy().item()

print(f"True energy: {true_energy:.4f} eV")
print(f"Predicted energy: {pred_energy:.4f} eV")

True energy: -257.5168 eV
Predicted energy: -68632240.0000 eV
